# Assignment Sesi 29 Tugas 1
Nama: Faraday Barr Fatahillah

**Tugas 1**

Setelah Anda mempelari cara untuk membaca dokumen PDF, Chunking, membuat metadata dan melakukan Hybrid Search. Sekarang coba gunakan data atau dokumen PDF `mitsubishi-combined-id.pdf`, untuk mencari konteks atau teks yang cocok dengan input atau query berikut:
- Detail spesifikasi Mitsubishi Destinator
- Mobil yang cocok untuk Travel dengan jumlah bangku atau seating capacity yang besar.
- Mobil untuk perjalanan jauh yang nyaman
- Mobil Mitsubishi yang irit bahan bakar
- Mobil Mitsubishi hybrid atau electric

Lakukan pencarian dengan Hybrid Search dan Vector Database yang digunakan adalah Pinecone dan ChromaDB.

In [13]:
# All import libraries
import re
import os
import chromadb
import numpy as np
from dotenv import load_dotenv
import pymupdf4llm
from tabulate import tabulate
from rank_bm25 import BM25Okapi
from chromadb.config import Settings
from pinecone import Pinecone, ServerlessSpec
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv()

True

### PDF Reading

In [ ]:
pdf_path = 'mitsubishi-combined-id.pdf'

In [15]:
md_text = pymupdf4llm.to_markdown(pdf_path, write_images=True)
md_text = re.sub(r'\n{3,}', '\n\n', md_text)
md_text = md_text.strip()

print('total karakter:', len(md_text))
print(md_text)

total karakter: 4604
## **PT Nusantara Telekomunikasi Digital (NTD)** 

## **1. Profil Perusahaan** 

PT Nusantara Telekomunikasi Digital (NTD) adalah perusahaan telekomunikasi nasional yang berdiri pada tahun 2008 dan berfokus pada penyediaan layanan jaringan seluler, internet broadband, serta solusi digital enterprise. Perusahaan ini beroperasi di lebih dari 150 kota di Indonesia dan memiliki lebih dari 35 juta pelanggan aktif. 

NTD menyediakan layanan 4G LTE dan telah mulai melakukan ekspansi jaringan 5G secara bertahap di kota-kota besar seperti Jakarta, Surabaya, Bandung, dan Medan. Selain layanan ritel, NTD juga memiliki divisi Business & Enterprise yang melayani kebutuhan konektivitas korporasi, data center, dan solusi cloud. 

## **Ringkasan Perusahaan** 

|**Kategori**|**Detail**|
|---|---|
|Tahun Berdiri|2008|
|Jumlah<br>Pelanggan|35+ juta|
|Cakupan Kota|150+ kota|
|Teknologi Utama|4G LTE, 5G|
|Segmen Bisnis|Retail & Enterprise|

## **2. Layanan Konsumen (Retail Services)** 

### Chunking Recursive

In [16]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
      separators=["\n## ", "\n# "]
)

chunks = splitter.split_text(md_text)

print("Total chunk:", len(chunks))
print("Contoh chunk pertama (total karakter):", len(chunks[0]),'\n')
print(chunks[0])

Total chunk: 14
Contoh chunk pertama (total karakter): 48 

## **PT Nusantara Telekomunikasi Digital (NTD)**


In [17]:
for i, chunk in enumerate(chunks):
    print(f"Chunk {i} (total karakter):", len(chunk))
    print(chunk)
    print("-" * 50)

Chunk 0 (total karakter): 48
## **PT Nusantara Telekomunikasi Digital (NTD)**
--------------------------------------------------
Chunk 1 (total karakter): 674

## **1. Profil Perusahaan** 

PT Nusantara Telekomunikasi Digital (NTD) adalah perusahaan telekomunikasi nasional yang berdiri pada tahun 2008 dan berfokus pada penyediaan layanan jaringan seluler, internet broadband, serta solusi digital enterprise. Perusahaan ini beroperasi di lebih dari 150 kota di Indonesia dan memiliki lebih dari 35 juta pelanggan aktif. 

NTD menyediakan layanan 4G LTE dan telah mulai melakukan ekspansi jaringan 5G secara bertahap di kota-kota besar seperti Jakarta, Surabaya, Bandung, dan Medan. Selain layanan ritel, NTD juga memiliki divisi Business & Enterprise yang melayani kebutuhan konektivitas korporasi, data center, dan solusi cloud. 

--------------------------------------------------
Chunk 2 (total karakter): 253
## **Ringkasan Perusahaan** 

|**Kategori**|**Detail**|
|---|---|
|Tahun Berdiri|2008

### Metadata & Hybrid Search

In [ ]:
def extract_metadata(chunk: str, chunk_index: int) -> dict:
    metadata = {
        "source": "mitsubishi-combined-id.pdf",
        "section": "General",
        "content_type": "text",
        "chunk_index": chunk_index,
        "chunk_length": len(chunk)
    }

    section_keywords = [
        ("XForce", "Mitsubishi XForce"),
        ("Xpander Cross", "Mitsubishi Xpander Cross"),
        ("Xpander", "Mitsubishi Xpander"),
        ("Spesifikasi", "Specifications"),
        ("Mesin", "Engine & Performance"),
        ("Keselamatan", "Safety"),
        ("Interior", "Interior & Comfort"),
        ("Eksterior", "Exterior Design"),
        ("Transmisi", "Transmission"),
        ("Garansi", "Warranty & Service"),
        ("Varian", "Variants"),
        ("Warna", "Color Options"),
        ("Aksesori", "Accessories"),
    ]
    for keyword, section_name in section_keywords:
        if keyword.lower() in chunk.lower():
            metadata["section"] = section_name
            break

    if "|" in chunk and chunk.count("|") >= 4:
        metadata["content_type"] = "table"
    elif chunk.strip().startswith("#"):
        metadata["content_type"] = "heading"

    return metadata


for i in range(min(3, len(chunks))):
    print(f"\n── Chunk {i} ──")
    print("Metadata:", extract_metadata(chunks[i], i))
    print("Teks:", chunks[i][:100], "...")


In [19]:

model = SentenceTransformer("intfloat/multilingual-e5-base")

def embed_passage(text: list[str]) -> np.ndarray:
    prefixed = [f'[PASSAGE] {t}' for t in text]
    return model.encode(prefixed, normalize_embeddings=True, show_progress_bar=True)

def embed_query(text: str) -> np.ndarray:
    prefixed = f'[QUERY] {text}'
    return model.encode(prefixed, normalize_embeddings=True, show_progress_bar=False)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7705.70it/s]


In [20]:
all_embeddings = embed_passage(chunks)

all_metadata = [extract_metadata(chunk, idx) for idx, chunk in enumerate(chunks)]
all_ids = [f"chunk_{idx}" for idx in range(len(chunks))]

print(f"Shape embeddings: {all_embeddings.shape}")
print(f"Number of chunks: {len(all_ids)}")

Batches: 100%|██████████| 1/1 [00:00<00:00,  1.19it/s]

Shape embeddings: (14, 768)
Number of chunks: 14


In [ ]:

client = chromadb.Client(Settings(persist_directory="./chroma_db"))

try:
    client.delete_collection("mitsubishi_docs")
except Exception as e:
    pass

collection = client.get_or_create_collection(
    name="mitsubishi_docs",
    metadata={"hnsw:space": "cosine"}
)

collection.add(
    ids=all_ids,
    documents=chunks,
    metadatas=all_metadata,
    embeddings=all_embeddings.tolist()
)

In [ ]:
pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))
PINECONE_INDEX_NAME = "mitsubishi-docs"

if PINECONE_INDEX_NAME not in pc.list_indexes().names():
    pc.create_index(
    name=PINECONE_INDEX_NAME, 
    dimension=all_embeddings.shape[1],
    metric="cosine",
    spec=ServerlessSpec(cloud="aws", region="us-east-1")
)

pine_index = pc.Index(PINECONE_INDEX_NAME)

In [23]:
BATCH_SIZE = 100

vectors_to_upsert = [
    (
        all_ids[i],
        all_embeddings[i].tolist(),
        {
            **all_metadata[i],
            "text": chunks[i][:500]
        }
    )
    for i in range(len(chunks))
]

for start in range(0, len(vectors_to_upsert), BATCH_SIZE):
    batch = vectors_to_upsert[start : start + BATCH_SIZE]
    pine_index.upsert(vectors=batch)
    print(f"  Upsert batch {start}–{start + len(batch) - 1} selesai")

  Upsert batch 0–13 selesai


In [24]:
def tokenize(text):
    return re.findall(r"\w+", text.lower())

tokenized_corpus = [tokenize(chunk) for chunk in chunks]
bm25 = BM25Okapi(tokenized_corpus)


In [29]:
def hybrid_search_chroma(query: str, k: int = 5) -> list[dict]:
    query_emb = embed_query(query)

    results = collection.query(
        query_embeddings=[query_emb.tolist()],
        n_results=10
    )

    vector_ids = results["ids"][0]
    distances = results["distances"][0]

    id_to_index = {doc_id: idx for idx, doc_id in enumerate(all_ids)}
    vector_scores = {
        id_to_index[vid]: 1 / (1 + d)
        for vid, d in zip(vector_ids, distances)
        if vid in id_to_index
    }

    tokenized_query = tokenize(query)
    bm25_scores = bm25.get_scores(tokenized_query)

    if np.max(bm25_scores) > 0: 
        bm25_normalized = bm25_scores / np.max(bm25_scores)
    else:
        bm25_normalized = bm25_scores


    combined_scores = {}

    for id, vscore in vector_scores.items():
        combined_scores[id] = vscore * 0.6

    for id, bscore in enumerate(bm25_normalized):
        combined_scores[id] = combined_scores.get(id, 0) + bscore * 0.4

    ranked = sorted(combined_scores.items(), key=lambda x: x[1], reverse=True)

    results = []
    for i, score in ranked[:k]:
        results.append({
            "chunk_id": i,
            "text": chunks[i],
            "metadata": all_metadata[i],
            "score": round(float(score), 4)
        })

    return results


In [30]:
def hybrid_search_pinecone(query: str, k: int = 5) -> list[dict]:
    query_emb = embed_query(query)

    pine_results = pine_index.query(
        vector = query_emb.tolist(),
        top_k = min(20, len(chunks)),
        include_metadata= True
    )

    id_to_index = {doc_id: idx for idx, doc_id in enumerate(all_ids)}
    vector_scores = {
        id_to_index[match["id"]]: match["score"]
        for match in pine_results["matches"]
        if match["id"] in id_to_index
    }

    tokenized_query = tokenize(query)
    bm25_raw        = bm25.get_scores(tokenized_query)

    bm25_max = np.max(bm25_raw)
    if bm25_max > 0:
        bm25_normalized = bm25_raw / bm25_max
    else:
        bm25_normalized = bm25_raw

    combined_scores = {}

    for chunk_id, vscore in vector_scores.items():
        combined_scores[chunk_id] = 0.6 * vscore

    for chunk_id, bscore in enumerate(bm25_normalized):
        combined_scores[chunk_id] = combined_scores.get(chunk_id, 0) + 0.4 * bscore

    ranked = sorted(combined_scores.items(), key=lambda x: x[1], reverse=True)

    results = []
    for chunk_id, score in ranked[:k]:
        results.append({
            "chunk_id" : chunk_id,
            "text" : chunks[chunk_id],
            "metadata" : all_metadata[chunk_id],
            "score" : round(float(score), 4)
        })

    return results

In [31]:
def print_results(query: str, results: list[dict], db_name: str = "") -> None:
    label = f"[{db_name}] " if db_name else ""
    print(f"\n{'='*70}")
    print(f"  {label}Query: {query!r}")
    print(f"{'='*70}")
    for i, res in enumerate(results, 1):
        print(f"\n Hasil #{i} | Score: {res['score']} | Section: {res['metadata']['section']}")
        print(f"Type: {res['metadata']['content_type']} | Chunk ID: {res['chunk_id']}")
        print(f"Teks: {res['text'][:300].strip()}...")
    print()

In [ ]:
queries = [
    "Detail spesifikasi Mitsubishi Destinator",
    "Mobil yang cocok untuk Travel dengan jumlah bangku atau seating capacity yang besar",
    "Mobil untuk perjalanan jauh yang nyaman",
    "Mobil Mitsubishi yang irit bahan bakar",
    "Mobil Mitsubishi hybrid atau electric"
]

print("Menjalankan Hybrid Search dengan ChromaDB...\n")
for query in queries:
    results = hybrid_search_chroma(query, k=3)
    print_results(query, results, db_name="ChromaDB")


In [ ]:
print("Menjalankan Hybrid Search dengan Pinecone...\n")
for query in queries:
    results = hybrid_search_pinecone(query, k=3)
    print_results(query, results, db_name="Pinecone")


In [ ]:
print("Perbandingan Top-1 Hasil: ChromaDB vs Pinecone\n")

table_data = []
for query in queries:
    res_chroma = hybrid_search_chroma(query, k=1)
    res_pine   = hybrid_search_pinecone(query, k=1)

    table_data.append([
        query[:40] + "...",
        res_chroma[0]['score'] if res_chroma else "-",
        res_chroma[0]['metadata']['section'] if res_chroma else "-",
        res_pine[0]['score'] if res_pine else "-",
        res_pine[0]['metadata']['section'] if res_pine else "-",
    ])

print(tabulate(
    table_data,
    headers=["Query", "ChromaDB Score", "ChromaDB Section", "Pinecone Score", "Pinecone Section"],
    tablefmt="grid"
))
